# Seance 10 — Facettage & Layering

Voir user guide :
- Layered & Multi-View Charts : https://altair-viz.github.io/user_guide/compound_charts.html
- Scale & Guide Resolution : https://altair-viz.github.io/user_guide/scale_resolve.html

## Objectifs
- Utiliser `facet` pour comparer des sous-groupes
- Utiliser `layer` pour superposer des elements
- Construire une tendance simple avec **pandas** (pas de transform Altair)

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

data_url = "https://raw.githubusercontent.com/datamisc/ts-2024/main/data.csv"
df = pd.read_csv(data_url, compression="gzip", low_memory=False)

## 1) Facettage : vote selon ideologie, separe par education

Question : la relation ideologie -> vote est-elle la meme selon le niveau d'education ?

Variables :
- `V241177` : ideologie (1-7)
- `V242096x` : vote presidentiel (Harris/Trump)
- `V241200` : education

On cree 3 groupes d'education pour que ce soit lisible en facettes.

In [ ]:
d = df.loc[
    df['V241177'].between(1, 7) & df['V242096x'].isin([1, 2]) & (df['V241200'] > 0),
    ['V241177', 'V242096x', 'V241200']
].copy()

d['ideologie'] = d['V241177'].astype(int)
d['vote_label'] = d['V242096x'].replace({1: 'Harris', 2: 'Trump'})

# Recodage simple education -> 3 groupes (a ajuster selon codebook)
d['edu_groupe'] = pd.cut(
    d['V241200'],
    bins=[0, 2, 4, 5],
    labels=['Education faible (1-2)', 'Education moyenne (3-4)', 'Education elevee (5)']
)

# Proportions vote par ideologie, dans chaque groupe d'education
prop = (
    d
    .groupby(['edu_groupe', 'ideologie', 'vote_label'], as_index=False)
    .size()
    .rename(columns={'size': 'n'})
)

tot = prop.groupby(['edu_groupe', 'ideologie'], as_index=False)['n'].sum().rename(columns={'n': 'n_total'})
prop = prop.merge(tot, on=['edu_groupe', 'ideologie'])
prop['proportion'] = prop['n'] / prop['n_total']

prop.head()

In [ ]:
alt.Chart(prop).mark_bar().encode(
    x=alt.X('ideologie', type='ordinal', title='Ideologie (1-7)'),
    y=alt.Y('proportion', type='quantitative', title='Proportion', axis=alt.Axis(format='%')),
    color=alt.Color('vote_label', type='nominal', title='Vote'),
    column=alt.Column('edu_groupe', type='nominal', title='Groupe education')
).properties(
    title=alt.TitleParams(
        text="Vote (Harris/Trump) selon ideologie, par groupe d'education",
        subtitle=[
            "Les profils ideologie->vote peuvent varier selon l'education.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    )
)

### Hack-Time 1 (10 min)

Essayez une autre variable de facette :
- facette par revenu (`V242025`, uniquement valeurs >= 0)
- ou facette par intention de vote (`V241042`)

Objectif : une grille lisible (pas trop de facettes).

In [ ]:
# Hack-Time 1 : votre code ici

## 2) Layering : scatter + tendance (binned mean)

On superpose :
- les points (Harris vs Trump)
- une ligne de tendance simple : moyenne de Harris par tranches de Trump

On calcule la tendance avec pandas (pas de transform Altair).

In [ ]:
d2 = df.loc[df['V241156'].between(0, 100) & df['V241157'].between(0, 100), ['V241156', 'V241157']].copy()

# Bins sur Trump
bins = list(range(0, 101, 10))
d2['trump_bin'] = pd.cut(d2['V241157'], bins=bins, right=False)
trend = (
    d2
    .groupby('trump_bin', as_index=False)['V241156']
    .mean()
    .rename(columns={'V241156': 'harris_moy'})
)

# Pour la ligne, on place x au centre du bin
trend['trump_bin_mid'] = trend['trump_bin'].apply(lambda i: (i.left + i.right) / 2)

trend.head()

In [ ]:
points = alt.Chart(d2.sample(2500, random_state=0)).mark_circle(size=20, opacity=0.25, color='#111827').encode(
    x=alt.X('V241157', type='quantitative', title='Thermometre Trump', scale=alt.Scale(domain=[0, 100])),
    y=alt.Y('V241156', type='quantitative', title='Thermometre Harris', scale=alt.Scale(domain=[0, 100]))
)

line = alt.Chart(trend).mark_line(color='#EF4444', size=3).encode(
    x=alt.X('trump_bin_mid', type='quantitative', title='Thermometre Trump (bins de 10)'),
    y=alt.Y('harris_moy', type='quantitative', title='Thermometre Harris (moyenne)', scale=alt.Scale(domain=[0, 100]))
)

alt.layer(points, line).properties(
    title=alt.TitleParams(
        text="Polarisation affective : points + tendance",
        subtitle=[
            "La ligne rouge resume la tendance moyenne (bins de 10).",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=420
)

### Hack-Time 2 (10 min)

Changez les bins de la tendance :
- bins de 5
- bins de 20

Que se passe-t-il sur la stabilite de la ligne ?

In [ ]:
# Hack-Time 2 : votre code ici